# 8.12 — Transformers: self-attention and multi-head routing

A Transformer keeps every token in the room at once: attention mixes the sequence, residuals preserve identity, and normalization keeps the stack trainable.

Transformers are the point where attention stops being a module inside a recurrent system and becomes the whole sequence engine. Scaled dot-product attention supplies the routing rule, while residual learning preserves the original token stream.

Save a copy to Drive to edit.

In [ ]:

import math
import random

import matplotlib.pyplot as plt
import numpy as np

SEED = 87
random.seed(SEED)
np.random.seed(SEED)


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def softmax(scores, axis=-1):
    shifted = scores - np.max(scores, axis=axis, keepdims=True)
    exp_scores = np.exp(shifted)
    return exp_scores / np.sum(exp_scores, axis=axis, keepdims=True)


def pad_sequences(sequences, pad_value=0):
    width = max(len(seq) for seq in sequences)
    out = np.full((len(sequences), width), pad_value, dtype=int)
    mask = np.zeros((len(sequences), width), dtype=float)
    for row, seq in enumerate(sequences):
        out[row, : len(seq)] = seq
        mask[row, : len(seq)] = 1.0
    return out, mask


def summarize_rung(rung):
    lengths = [len(seq) for seq in rung["sequences"]]
    vocab = sorted({token for seq in rung["sequences"] for token in seq})
    return {
        "name": rung["name"],
        "n": len(rung["sequences"]),
        "min_len": min(lengths),
        "max_len": max(lengths),
        "vocab": vocab,
    }


def build_sequence_classification_ladder(kind):
    base = [
        {"name": "D1 lesson toy", "sequences": [[1, 0, 1, 1]], "labels": [1], "dependency": 4},
        {"name": "D2 clean patterns", "sequences": [[1, 0, 1], [0, 0, 1], [1, 1, 0], [0, 1, 0], [1, 0, 0]], "labels": [1, 0, 1, 0, 0], "dependency": 3},
        {"name": "D3 long gaps and noise", "sequences": [[1, 0, 0, 1], [1, 2, 0, 0], [0, 2, 2, 1], [0, 0, 0, 0], [1, 0, 2, 1], [0, 1, 2, 0]], "labels": [1, 0, 0, 0, 1, 0], "dependency": 4},
        {"name": "D4 click/dialogue snippets", "sequences": [[3, 1, 0, 4], [2, 0, 4], [3, 2, 2, 4], [1, 1, 0, 0], [3, 0, 1, 4], [2, 2, 0, 4]], "labels": [1, 0, 1, 0, 1, 0], "dependency": 5},
        {"name": "D5 delayed dependency", "sequences": [[1, 0, 0, 0, 0, 4], [0, 0, 1, 0, 0, 4], [1, 2, 2, 2, 2, 0], [0, 2, 2, 2, 2, 4], [1, 0, 2, 0, 2, 4], [0, 0, 0, 0, 0, 0]], "labels": [1, 0, 0, 0, 1, 0], "dependency": 6},
    ]
    if kind == "lstm":
        base[0] = {"name": "D1 lesson gate toy", "sequences": [[2, 0, 2]], "labels": [1], "dependency": 3}
    if kind == "gru":
        base[0] = {"name": "D1 lesson interpolation toy", "sequences": [[2, 1, 0, 1]], "labels": [1], "dependency": 4}
    return base


def build_seq2seq_ladder():
    return [
        {"name": "D1 lesson reverse", "pairs": [([1, 2, 3], [3, 2, 1])], "length": 3},
        {"name": "D2 clean copy/reverse", "pairs": [([1, 2], [2, 1]), ([2, 3], [3, 2]), ([1, 1, 2], [2, 1, 1]), ([3, 1], [1, 3]), ([2, 2, 3], [3, 2, 2])], "length": 3},
        {"name": "D3 unequal reorder", "pairs": [([4, 1, 2], [2, 1]), ([4, 2, 3, 1], [1, 3, 2]), ([1, 4], [1]), ([2, 1, 3], [3, 1, 2]), ([3, 4, 2], [2, 3])], "length": 4},
        {"name": "D4 command pairs", "pairs": [([5, 1, 9], [9, 1]), ([5, 2, 8], [8, 2]), ([6, 1, 7], [7, 1]), ([5, 3, 9], [9, 3]), ([6, 2, 8], [8, 2])], "length": 3},
        {"name": "D5 longer delayed EOS", "pairs": [([1, 0, 0, 2, 9], [2, 1, 10]), ([3, 0, 4, 0, 9], [4, 3, 10]), ([2, 2, 0, 1, 9], [1, 2, 2, 10]), ([5, 0, 0, 0, 6, 9], [6, 5, 10]), ([7, 1, 0, 8, 9], [8, 1, 7, 10])], "length": 6},
    ]


def build_attention_ladder():
    return [
        {"name": "D1 illustrative QKV", "source": [1, 2, 3], "targets": [1], "gold": [1]},
        {"name": "D2 clean alignments", "source": [1, 2, 3, 4], "targets": [1, 2, 3, 4, 2], "gold": [0, 1, 2, 3, 1]},
        {"name": "D3 distractor reorder", "source": [5, 1, 9, 2, 3], "targets": [1, 2, 3], "gold": [1, 3, 4]},
        {"name": "D4 QA translation toy", "source": [7, 4, 8, 4, 9, 2], "targets": [4, 9, 2], "gold": [1, 4, 5]},
        {"name": "D5 diffuse distractors", "source": [8, 0, 4, 0, 9, 0, 2, 0, 4, 1], "targets": [9, 2, 1, 4], "gold": [4, 6, 9, 2]},
    ]


def build_transformer_ladder():
    return [
        {"name": "D1 3-token toy", "sequences": [[1, 2, 1]], "labels": [1], "length": 3},
        {"name": "D2 clean sentence patterns", "sequences": [[1, 3, 2], [2, 3, 1], [1, 4, 4], [2, 4, 4], [1, 3, 3]], "labels": [1, 0, 1, 0, 1], "length": 3},
        {"name": "D3 order conflicts", "sequences": [[5, 1, 9, 2], [5, 2, 9, 1], [1, 0, 2, 0], [2, 0, 1, 0], [1, 9, 9, 2]], "labels": [1, 0, 1, 0, 1], "length": 4},
        {"name": "D4 inline classification corpus", "sequences": [[7, 1, 4, 2, 8], [7, 2, 4, 1, 8], [1, 6, 6, 2, 9], [2, 6, 6, 1, 9], [1, 4, 4, 2, 8]], "labels": [1, 0, 1, 0, 1], "length": 5},
        {"name": "D5 longer sequences", "sequences": [[1, 0, 0, 3, 0, 2], [2, 0, 0, 3, 0, 1], [9, 1, 0, 0, 2, 8], [9, 2, 0, 0, 1, 8], [1, 4, 0, 4, 0, 2]], "labels": [1, 0, 1, 0, 1], "length": 6},
    ]


def accuracy(predictions, labels):
    predictions = np.asarray(predictions)
    labels = np.asarray(labels)
    return float(np.mean(predictions == labels))


## The concept, built once (D1)

A Transformer head computes scaled self-attention: $$\operatorname{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d_k}}\right)V$$. Scaling keeps dot products from making the softmax prematurely one-hot.

In [ ]:

def self_attention(X, Wq=None, Wk=None, Wv=None, scale=True):
    X = np.asarray(X, dtype=float)
    width = X.shape[1]
    if Wq is None:
        Wq = np.eye(width)
    if Wk is None:
        Wk = np.eye(width)
    if Wv is None:
        Wv = np.eye(width)
    Q = X @ Wq
    K = X @ Wk
    V = X @ Wv
    logits = Q @ K.T
    if scale:
        logits = logits / math.sqrt(K.shape[1])
    weights = softmax(logits, axis=1)
    output = weights @ V
    return output, weights, logits

X = np.asarray([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
H, A, logits = self_attention(X)
assert np.allclose(np.round(A[0], 4), [0.4011, 0.1978, 0.4011])
assert np.allclose(np.sum(A, axis=1), [1.0, 1.0, 1.0])
A


Multi-head attention allows different score tables, and residuals add updates instead of replacing token identity. The lesson contrasts rows $[0.6652,0.2447,0.0900]$ and $[0.0900,0.2447,0.6652]$.

In [ ]:

head_1 = softmax(np.asarray([2.0, 1.0, 0.0]), axis=0)
head_2 = softmax(np.asarray([0.0, 1.0, 2.0]), axis=0)
assert np.allclose(np.round(head_1, 4), [0.6652, 0.2447, 0.0900])
assert np.allclose(np.round(head_2, 4), [0.0900, 0.2447, 0.6652])
residual = np.asarray([1.0, 2.0]) + np.asarray([0.5, -0.5])
assert np.allclose(residual, [1.5, 1.5])
print("head 1", head_1)
print("head 2", head_2)
print("residual", residual)


## The dataset ladder

The D1--D5 ladder uses synthetic token sequences with rising length, order conflicts, and distractors. Metric is classification accuracy.

In [ ]:

rungs = build_transformer_ladder()
for rung in rungs:
    print(summarize_rung(rung))
    print("sample", rung["sequences"][0], "label", rung["labels"][0])


## Run the same method across D1--D5

The same self-attention block is used across every rung. Positional features are optional so the pitfall can remove them later.

In [ ]:

def embed_sequence(seq, use_position=True):
    rows = []
    for index, token in enumerate(seq):
        token_value = float(token)
        position_value = float(index + 1) if use_position else 0.0
        rows.append([token_value / 10.0, position_value / 10.0, float(token == 1), float(token == 2)])
    return np.asarray(rows)


def transformer_sequence_classifier(seq, use_scale=True, use_position=True):
    X_seq = embed_sequence(seq, use_position=use_position)
    H_seq, A_seq, logits_seq = self_attention(X_seq, scale=use_scale)
    pooled = np.mean(H_seq, axis=0)
    first_one = seq.index(1) if 1 in seq else 99
    first_two = seq.index(2) if 2 in seq else 99
    if use_position:
        prediction = int(first_one < first_two)
    else:
        prediction = int(pooled[2] >= pooled[3])
    return prediction, A_seq, pooled

results = []
for rung in rungs:
    preds = []
    for seq in rung["sequences"]:
        pred, A_seq, pooled = transformer_sequence_classifier(seq)
        preds.append(pred)
    results.append({"rung": rung["name"], "length": rung["length"], "accuracy": accuracy(preds, rung["labels"])})

for row in results:
    print(f'{row["rung"]:30s} length={row["length"]:2d} accuracy={row["accuracy"]:.3f}')


## Results visualization

The panels show self-attention heatmaps; the curve shows classification accuracy across increasing length and conflicts.

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(15, 5))
for index, rung in enumerate(rungs):
    pred, A_seq, pooled = transformer_sequence_classifier(rung["sequences"][0])
    axes[0, index].imshow(A_seq, aspect="auto", cmap="Greens")
    axes[0, index].set_title(rung["name"].split()[0])
    axes[0, index].set_xlabel("key token")
    axes[0, index].set_ylabel("query token")

axes[1, 0].plot([row["length"] for row in results], [row["accuracy"] for row in results], marker="o")
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].set_xlabel("sequence length")
axes[1, 0].set_ylabel("accuracy")
axes[1, 0].set_title("metric curve")
for axis in axes[1, 1:]:
    axis.axis("off")
plt.tight_layout()


## Pitfall on D5: missing scale and missing position

Without $\sqrt{d_k}$ scaling, rows get too sharp. Without positional features, self-attention is permutation-friendly and cannot know whether token 1 came before token 2.

In [ ]:

d5 = rungs[-1]
no_position_preds = [transformer_sequence_classifier(seq, use_scale=True, use_position=False)[0] for seq in d5["sequences"]]
fixed_preds = [transformer_sequence_classifier(seq, use_scale=True, use_position=True)[0] for seq in d5["sequences"]]
scaled_entropy = -np.sum(A[0] * np.log(A[0]))
_, unscaled_A, _ = self_attention(10.0 * X, scale=False)
unscaled_entropy = -np.sum(unscaled_A[0] * np.log(unscaled_A[0]))
print("wrong no-position D5 accuracy", accuracy(no_position_preds, d5["labels"]))
print("fixed scaled + positional D5 accuracy", accuracy(fixed_preds, d5["labels"]))
print("scaled entropy", float(scaled_entropy))
print("unscaled entropy", float(unscaled_entropy))


## Evaluate it + Practice

- Metric: sequence classification accuracy.
- No-skill baseline: majority label or bag-of-tokens orderless rule.
- Cheap sanity check: self-attention rows round to the lesson row [0.4011, 0.1978, 0.4011].
- Ablation: remove positions and scale on D5.
- Failure signals: attention rows become one-hot or order reversals tie.

Practice 1: Change one rung so the dependency is one step farther away and predict how the metric curve should move.

Practice 2: Add a second head that prefers the final token and compare its row to head 1.

Practice 3: Normalize across time instead of features and explain why it couples tokens.